# Stage 4: Features (text → vectors)

**Stage:** 4 (features) · **Runs on:** Kaggle and locally (CPU)

| | |
|---|---|
| Input | the Stage 2 split files |
| Output | per dataset (`liar`, `welfake`, `covid`, `fakenewsnet`) and for LIAR regression (`liar_reg`): `features__<key>.joblib` (the fitted feature builder), `X_tfidf__<key>__<split>.npz`, `X_dense__<key>__<split>.npy` |

Every text becomes two **vectors**:

| Vector | Length | Used by |
|---|---|---|
| TF-IDF | one number per word / word pair in the vocabulary (sparse: mostly zeros) | Logistic Regression, Complement NB, Linear SVM, Ridge |
| Dense | 300 SVD numbers (a compressed TF-IDF) + 8 style numbers | LightGBM, LightGBM regressor, clustering |

Each feature builder is **fitted on its dataset's training part only**, then applied to the
validation and test parts. Row *i* of a matrix is row *i* of the matching split file.
No model is trained here and no labels are used: Stage 5 does the training.
DistilBERT does not use these vectors; it reads the text itself.

**On Kaggle:** attach the `misinfo_utils` utility script and add these notebooks as inputs (**Add Input → Your Work → Notebooks**): `02_split`.

In [5]:
# Setup: import the shared code. Works on Kaggle and locally.
import sys
from pathlib import Path

# Locally, misinfo_utils.py is in <project>/src/. On Kaggle it is an attached Utility Script,
# so this loop finds nothing and the import below still works.
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "src" / "misinfo_utils.py").exists():
        sys.path.insert(0, str(folder / "src"))
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import misinfo_utils as mu

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 100)
paths = mu.setup("04")

misinfo_utils | stage 04 | running on local
  read from : D:\misinfo-project\data\raw, D:\misinfo-project\outputs
  write to  : D:\misinfo-project\outputs\04_features


In [6]:
SVD_DIMS = 300          # length of the compressed vector
MAX_FEATURES = 200_000  # vocabulary limit (most frequent words / word pairs)

In [9]:
import time, joblib
from scipy import sparse

summary = []
for key in mu.FEATURE_KEYS:
    folder = f"{key}_preprocessed"          # e.g. covid_preprocessed
    t0 = time.time()
    parts = {s: mu.load_split(key, s) for s in mu.SPLITS}
    builder = mu.FeatureBuilder(svd_dims=SVD_DIMS, max_features=MAX_FEATURES).fit(parts["train"]["text"])
    joblib.dump(builder, mu.output_path(mu.builder_file(key), subfolder=folder))

    for s, df in parts.items():
        X_sparse = builder.sparse(df["text"])
        X_dense = builder.dense(df["text"], X_sparse)
        assert X_sparse.shape[0] == X_dense.shape[0] == len(df)      # one vector per row
        sparse.save_npz(mu.output_path(mu.tfidf_file(key, s), subfolder=folder), X_sparse)
        np.save(mu.output_path(mu.dense_file(key, s), subfolder=folder), X_dense)

    summary.append({"key": key, "train rows": len(parts["train"]),
                    "vocabulary (TF-IDF length)": len(builder.tfidf.vocabulary_),
                    "dense length": builder.n_svd + len(mu.STYLE_FEATURE_NAMES),
                    "share of TF-IDF kept by SVD": round(float(builder.svd.explained_variance_ratio_.sum()), 3),
                    "seconds": round(time.time() - t0, 1)})
    print(f"{key:12s} done in {time.time() - t0:.0f}s")

pd.DataFrame(summary).set_index("key")

liar         done in 4s
welfake      done in 131s
covid        done in 6s
fakenewsnet  done in 4s
liar_reg     done in 5s


,train rows,vocabulary (TF-IDF length),dense length,share of TF-IDF kept by SVD,seconds
key,,,,,
liar,7114,18546,308,0.265,4.3
welfake,14000,200000,308,0.123,131.0
covid,7195,26988,308,0.267,6.1
fakenewsnet,7243,12999,308,0.298,3.5
liar_reg,8953,22827,308,0.245,5.4


## What a vector looks like (one example)

In [10]:
example = mu.load_split("covid", "train").iloc[0]
builder = mu.load_builder("covid")
vec = builder.sparse([example["text"]])
vocab = builder.tfidf.get_feature_names_out()
nonzero = vec.nonzero()[1]
print("Text:", example["text"][:200])
print(f"TF-IDF vector: {vec.shape[1]:,} numbers, {len(nonzero)} of them non-zero. Largest:")
print(pd.Series(vec.toarray()[0, nonzero], index=vocab[nonzero]).nlargest(8).round(3).to_string())
dense = builder.dense([example["text"]])[0]
print(f"\nDense vector: {len(dense)} numbers. First 5 SVD values: {np.round(dense[:5], 3)}")
print("Style values:", dict(zip(mu.STYLE_FEATURE_NAMES, np.round(dense[-8:], 2))))

Text: Complete fibrinolysis shutdown seen in severe .
TF-IDF vector: 26,988 numbers, 7 of them non-zero. Largest:
in severe    0.489
shutdown     0.473
seen in      0.422
complete     0.376
severe       0.325
seen         0.317
in           0.103

Dense vector: 308 numbers. First 5 SVD values: [ 0.03  -0.013 -0.003  0.009 -0.006]
Style values: {'log_words': np.float64(-1.98), 'log_chars': np.float64(-1.9), 'exclamations': np.float64(-0.17), 'questions': np.float64(-0.17), 'caps_ratio': np.float64(-0.68), 'digit_ratio': np.float64(-0.62), 'allcaps_word_ratio': np.float64(-0.58), 'quote_ratio': np.float64(-0.2)}
